In [3]:
import pandas as pd
from collections import defaultdict

# Load match data
df = pd.read_csv('wta_matches_2023.csv')
df = df.sort_values('tourney_date')

# Initialize Elo ratings
elo_ratings = defaultdict(lambda: 1500)

# Store Elo snapshot *before* each match
elo_snapshots = []

K = 30

def win_prob(rating_i, rating_j):
    return 1 / (1 + 10 ** ((rating_j - rating_i) / 400))

for _, match in df.iterrows():
    winner = match['winner_name']
    loser = match['loser_name']

    rating_winner = elo_ratings[winner]
    rating_loser = elo_ratings[loser]

    # Record pre-match Elo ratings
    elo_snapshots.append({
        'date': match['tourney_date'],
        'tournament': match['tourney_name'],
        'winner': winner,
        'loser': loser,
        'winner_elo_before': rating_winner,
        'loser_elo_before': rating_loser
    })

    # Calculate expected outcomes
    expected_win = win_prob(rating_winner, rating_loser)
    expected_loss = 1 - expected_win

    # Update ratings
    elo_ratings[winner] += K * (1 - expected_win)
    elo_ratings[loser] += K * (0 - expected_loss)

# Create DataFrame
elo_df = pd.DataFrame(elo_snapshots)
elo_df

,date,tournament,winner,loser,winner_elo_before,loser_elo_before
0,20230102,United Cup,Jessica Pegula,Martina Trevisan,1500.000000,1500.000000
1,20230102,Auckland,Rebeka Masarova,Ysaline Bonaventure,1500.000000,1500.000000
2,20230102,Auckland,Coco Gauff,Danka Kovinic,1500.000000,1500.000000
3,20230102,Auckland,Coco Gauff,Rebeka Masarova,1515.000000,1515.000000
4,20230102,Adelaide 1,Liudmila Samsonova,Shuai Zhang,1500.000000,1500.000000
...,...,...,...,...,...,...
2805,20231110,BJK Cup Playoffs,Nao Hibino,Yuliana Lizarazo,1467.484847,1500.000000
2806,20231110,BJK Cup Playoffs,Camila Osorio,Mai Hontama,1548.989285,1594.023540
2807,20231110,BJK Cup Playoffs,Yanina Wickmayer,Dalma Galfi,1533.074031,1477.102815
2808,20231110,BJK Cup Playoffs,Fernanda Contreras Gomez,Tamira Paszek,1462.639496,1514.335303


In [4]:
elo_df['difference'] = elo_df['winner_elo_before'] - elo_df['loser_elo_before']
elo_df

,date,tournament,winner,loser,winner_elo_before,loser_elo_before,difference
0,20230102,United Cup,Jessica Pegula,Martina Trevisan,1500.000000,1500.000000,0.000000
1,20230102,Auckland,Rebeka Masarova,Ysaline Bonaventure,1500.000000,1500.000000,0.000000
2,20230102,Auckland,Coco Gauff,Danka Kovinic,1500.000000,1500.000000,0.000000
3,20230102,Auckland,Coco Gauff,Rebeka Masarova,1515.000000,1515.000000,0.000000
4,20230102,Adelaide 1,Liudmila Samsonova,Shuai Zhang,1500.000000,1500.000000,0.000000
...,...,...,...,...,...,...,...
2805,20231110,BJK Cup Playoffs,Nao Hibino,Yuliana Lizarazo,1467.484847,1500.000000,-32.515153
2806,20231110,BJK Cup Playoffs,Camila Osorio,Mai Hontama,1548.989285,1594.023540,-45.034255
2807,20231110,BJK Cup Playoffs,Yanina Wickmayer,Dalma Galfi,1533.074031,1477.102815,55.971216
2808,20231110,BJK Cup Playoffs,Fernanda Contreras Gomez,Tamira Paszek,1462.639496,1514.335303,-51.695807


In [5]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Randomly assign winner to be Player A or B
winner_is_A = np.random.randint(0, 2, size=len(elo_df)) == 1

# Create transformed DataFrame
elo_transformed = pd.DataFrame({
    'date': elo_df['date'],
    'tournament': elo_df['tournament'],
    
    'player_A_name': np.where(winner_is_A, elo_df['winner'], elo_df['loser']),
    'player_B_name': np.where(winner_is_A, elo_df['loser'], elo_df['winner']),
    
    'player_A_elo_before': np.where(winner_is_A, elo_df['winner_elo_before'], elo_df['loser_elo_before']),
    'player_B_elo_before': np.where(winner_is_A, elo_df['loser_elo_before'], elo_df['winner_elo_before']),
    
    'player_A_win': winner_is_A.astype(int)
})
elo_transformed

,date,tournament,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win
0,20230102,United Cup,Martina Trevisan,Jessica Pegula,1500.000000,1500.000000,0
1,20230102,Auckland,Rebeka Masarova,Ysaline Bonaventure,1500.000000,1500.000000,1
2,20230102,Auckland,Danka Kovinic,Coco Gauff,1500.000000,1500.000000,0
3,20230102,Auckland,Rebeka Masarova,Coco Gauff,1515.000000,1515.000000,0
4,20230102,Adelaide 1,Shuai Zhang,Liudmila Samsonova,1500.000000,1500.000000,0
...,...,...,...,...,...,...,...
2805,20231110,BJK Cup Playoffs,Yuliana Lizarazo,Nao Hibino,1500.000000,1467.484847,0
2806,20231110,BJK Cup Playoffs,Mai Hontama,Camila Osorio,1594.023540,1548.989285,0
2807,20231110,BJK Cup Playoffs,Dalma Galfi,Yanina Wickmayer,1477.102815,1533.074031,0
2808,20231110,BJK Cup Playoffs,Tamira Paszek,Fernanda Contreras Gomez,1514.335303,1462.639496,0


In [6]:
elo_transformed['elo_difference'] = elo_transformed['player_A_elo_before'] - elo_transformed['player_B_elo_before']
elo_transformed

,date,tournament,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference
0,20230102,United Cup,Martina Trevisan,Jessica Pegula,1500.000000,1500.000000,0,0.000000
1,20230102,Auckland,Rebeka Masarova,Ysaline Bonaventure,1500.000000,1500.000000,1,0.000000
2,20230102,Auckland,Danka Kovinic,Coco Gauff,1500.000000,1500.000000,0,0.000000
3,20230102,Auckland,Rebeka Masarova,Coco Gauff,1515.000000,1515.000000,0,0.000000
4,20230102,Adelaide 1,Shuai Zhang,Liudmila Samsonova,1500.000000,1500.000000,0,0.000000
...,...,...,...,...,...,...,...,...
2805,20231110,BJK Cup Playoffs,Yuliana Lizarazo,Nao Hibino,1500.000000,1467.484847,0,32.515153
2806,20231110,BJK Cup Playoffs,Mai Hontama,Camila Osorio,1594.023540,1548.989285,0,45.034255
2807,20231110,BJK Cup Playoffs,Dalma Galfi,Yanina Wickmayer,1477.102815,1533.074031,0,-55.971216
2808,20231110,BJK Cup Playoffs,Tamira Paszek,Fernanda Contreras Gomez,1514.335303,1462.639496,0,51.695807


In [8]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [10]:
%%R

require('tidyverse')
require('DescTools')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: DescTools


In [11]:
%%R -i elo_transformed

logistic <- glm(player_A_win ~ player_A_elo_before + player_B_elo_before, data=elo_transformed, family="binomial")

print(summary(logistic))
print(PseudoR2(logistic, which="McFadden"))


Call:
glm(formula = player_A_win ~ player_A_elo_before + player_B_elo_before, 
    family = "binomial", data = elo_transformed)

Coefficients:
                      Estimate Std. Error z value Pr(>|z|)    
(Intercept)          0.5480552  1.1054310   0.496     0.62    
player_A_elo_before  0.0072320  0.0006507  11.115   <2e-16 ***
player_B_elo_before -0.0076025  0.0006566 -11.578   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 3895.2  on 2809  degrees of freedom
Residual deviance: 3675.2  on 2807  degrees of freedom
AIC: 3681.2

Number of Fisher Scoring iterations: 4

  McFadden 
0.05650104 
